In [1]:
# Updated AlphaGenome code to perform extra predictions
# Setup working environment and import libraries, DNA models
from alphagenome.data import gene_annotation, genome, transcript, track_data
from alphagenome.models import dna_client, variant_scorers
from alphagenome.visualization import plot_components
import pandas as pd
import itables
from itables import show
from io import StringIO
from tqdm import tqdm
import csv
import numpy as np
import matplotlib.pyplot as plt

# Load the model.
API_KEY = 'dna_model = dna_client.create(API_KEY)

HG38_GTF_FEATHER = (
    'https://storage.googleapis.com/alphagenome/reference/gencode/'
    'hg38/gencode.v46.annotation.gtf.gz.feather'
)
MM10_GTF_FEATHER = (
    'https://storage.googleapis.com/alphagenome/reference/gencode/'
    'mm10/gencode.vM23.annotation.gtf.gz.feather'
)

# Initialize an empty dictionary to serve as a variant effect prediction cache.
_prediction_cache = {}

_transcript_extractor_cache = {}

In [2]:
# Load the DNA models and view gene ontology
output_metadata = dna_model.output_metadata(
    dna_client.Organism.HOMO_SAPIENS
).concatenate()

In [ ]:
# Load gene annotations (from GENCODE).
gtf = pd.read_feather(
    'https://storage.googleapis.com/alphagenome/reference/gencode/'
    'hg38/gencode.v46.annotation.gtf.gz.feather'
)

# Filter to protein-coding genes and highly supported transcripts.
gtf_transcript = gene_annotation.filter_transcript_support_level(
    gene_annotation.filter_protein_coding(gtf), ['1']
)

# Extractor for identifying transcripts in a region.
transcript_extractor = transcript.TranscriptExtractor(gtf_transcript)

# Also define an extractor that fetches only the longest transcript per gene.
gtf_longest_transcript = gene_annotation.filter_to_longest_transcript(
    gtf_transcript
)
longest_transcript_extractor = transcript.TranscriptExtractor(
    gtf_longest_transcript
)

In [ ]:
# Prediction #0: the 9p21 deletion

In [ ]:
# Predict sequence
# Input the ref sequence (524288bp)
ref_string = """...
"""
# Input the alt sequence (524288bp with deletion sequence masked as Ns)
alt_string = """...
"""

# Remove end breaks for the long strings
def clean_sequence(text: str) -> str:
    return "".join(text.split())
cleaned_ref = clean_sequence(ref_string)
cleaned_alt = clean_sequence(alt_string)
print(len(cleaned_ref))
print(len(cleaned_alt))


In [ ]:
# Define function to add Ns to the right side of the sequence
def pad_right(seq: str, target_len: int) -> str:
    seq = "".join(seq.split()).upper()  # clean sequence
    pad_len = max(0, target_len - len(seq))
    return seq + "N" * pad_len

In [ ]:
# Run predict_sequence function for Ref and Alt sequences
ref_prediction = dna_model.predict_sequence(
    #sequence=cleaned_ref.center(dna_client.SEQUENCE_LENGTH_1MB,'N'),
    sequence=pad_right(cleaned_ref, dna_client.SEQUENCE_LENGTH_500KB), # Pad to valid sequence length to the right of sequence
    organism=dna_client.Organism.HOMO_SAPIENS,
    requested_outputs={dna_client.OutputType.RNA_SEQ,
                       dna_client.OutputType.CHIP_HISTONE,
                       dna_client.OutputType.DNASE,
                       dna_client.OutputType.CHIP_TF},
    ontology_terms=['CL:1001606','CL:2000045','CL:1001608','CL:0000312','CL:1000458','CL:0002551',
                  'UBERON:0002097','UBERON:0004264','EFO:0005720','EFO:0002103'],  
    interval=genome.Interval('chr9',21756689, 22280977).resize(dna_client.SEQUENCE_LENGTH_500KB) #genome interval for prediction
)
alt_prediction = dna_model.predict_sequence(
    #sequence=cleaned_alt.center(dna_client.SEQUENCE_LENGTH_1MB,'N'),
    sequence=pad_right(cleaned_alt, dna_client.SEQUENCE_LENGTH_500KB),  # Pad to valid sequence length to the right of sequence
    organism=dna_client.Organism.HOMO_SAPIENS,
    requested_outputs={dna_client.OutputType.RNA_SEQ,
                       dna_client.OutputType.CHIP_HISTONE,
                       dna_client.OutputType.DNASE,
                       dna_client.OutputType.CHIP_TF},
    ontology_terms=['CL:1001606','CL:2000045','CL:1001608','CL:0000312','CL:1000458','CL:0002551',
                  'UBERON:0002097','UBERON:0004264','EFO:0005720','EFO:0002103'],  
    interval=genome.Interval('chr9',21756689, 22280977).resize(dna_client.SEQUENCE_LENGTH_500KB) 
)

In [ ]:
# Define plotting parameters
interval=genome.Interval('chr9',21756689, 22280977).resize(dna_client.SEQUENCE_LENGTH_500KB)
highlight = genome.Interval("chr9", 22180210, 22280977)

In [ ]:
# Plotting
transcripts = transcript_extractor.extract(interval)
_ = plot_components.plot(
    [
        plot_components.TranscriptAnnotation(transcripts, fig_height=2),
       # RNA-seq tracks
        plot_components.Tracks(
            tdata=alt_prediction.rna_seq.filter_to_negative_strand()
            - ref_prediction.rna_seq.filter_to_negative_strand(),
            ylabel_template='{biosample_name} ({strand})\n{name}',
            filled=True,
            shared_y_scale=True,
            track_colors="#4C72B0",
        ),
               ],
    interval=genome.Interval(
    chromosome='chr9', start=21965000, end=22015000
    ),
    title=(
        'RNA seq: alt - ref (predict_sequence)'
    ),
)

# Save the plot
plt.savefig('Deletion_AlphaGenome_RNAseq_multipleontologies_predictsequence.pdf')

In [ ]:
# Plotting (only plot the ontologies with Chip histone available)
ontologies = [
    "CL:0000312",
    "CL:0002551",
    "CL:1001606",
    "CL:1001608",
    "CL:2000045",
]

mask = ref_prediction.rna_seq.metadata["ontology_curie"].isin(ontologies).to_numpy()

ref_subset = ref_prediction.rna_seq.filter_tracks(mask)
alt_subset = alt_prediction.rna_seq.filter_tracks(mask)

transcripts = transcript_extractor.extract(interval)
_ = plot_components.plot(
    [
        plot_components.TranscriptAnnotation(transcripts, fig_height=2),
       # RNA-seq tracks
        plot_components.Tracks(
            tdata=alt_subset.filter_to_negative_strand()
            - ref_subset.filter_to_negative_strand(),
            ylabel_template='{biosample_name} ({strand})\n{name}',
            filled=True,
            shared_y_scale=True,
            track_colors="#4C72B0",
        ),
               ],
    interval=genome.Interval(
    chromosome='chr9', start=21965000, end=22015000
    ),
    title=(
        'RNA seq: alt - ref (predict_sequence)'
    ),
)

# Save the plot
plt.savefig('Deletion_AlphaGenome_RNAseq_multipleontologies.pdf',bbox_inches='tight')

In [ ]:
# Plotting histone chip seq - zoom in to CDKN2A/2B (H3K4me3)

transcripts = transcript_extractor.extract(interval)
#alt_h3k4me3 = alt_prediction.chip_histone.metadata.histone_mark == "H3K4me3"
mask1 = (ref_prediction.chip_histone.metadata.histone_mark == "H3K4me3").to_numpy()
mask2 = (alt_prediction.chip_histone.metadata.histone_mark == "H3K4me3").to_numpy()
# filter track
ref_h3k4me3 = ref_prediction.chip_histone.filter_tracks(mask1)
alt_h3k4me3 = alt_prediction.chip_histone.filter_tracks(mask2)

_ = plot_components.plot(
    [
        plot_components.TranscriptAnnotation(transcripts, fig_height=2),
        plot_components.Tracks(
            tdata=alt_h3k4me3 - ref_h3k4me3,
            ylabel_template="{biosample_name} ({strand})\n{name}",
            filled=True,
            shared_y_scale=True,
            track_colors="#C44E52",
        ),
    ],
    interval=genome.Interval(
        chromosome="chr9", start=21965000, end=22015000
    ),
    title="ChIP Histone: H3K4me3 + H3K9ac alt - ref (predict_sequence)",
)
# Save the plot
plt.savefig('Deletion_AlphaGenome_ChipHistone_H3K4me3.pdf')

In [ ]:
# Plotting histone chip seq - zoom in to CDKN2A/2B (H3K9ac)

transcripts = transcript_extractor.extract(interval)
mask3 = (ref_prediction.chip_histone.metadata.histone_mark == "H3K9ac").to_numpy()
mask4 = (alt_prediction.chip_histone.metadata.histone_mark == "H3K9ac").to_numpy()
# filter track
ref_h3k9ac = ref_prediction.chip_histone.filter_tracks(mask3)
alt_h3k9ac = alt_prediction.chip_histone.filter_tracks(mask4)

_ = plot_components.plot(
    [
        plot_components.TranscriptAnnotation(transcripts, fig_height=2),
        plot_components.Tracks(
            tdata=alt_h3k9ac - ref_h3k9ac,
            ylabel_template="{biosample_name} ({strand})\n{name}",
            filled=True,
            shared_y_scale=True,
            track_colors="#DD8452",
        ),
    ],
    interval=genome.Interval(
        chromosome="chr9", start=21965000, end=22015000
    ),
    annotations=[
        plot_components.IntervalAnnotation(
            [highlight],
            colors="orange",
            alpha=0.3,
            labels=["9p21 deletion"],
        )
    ],
    title="ChIP Histone: H3K9ac alt - ref (predict_sequence)",
)
# Save the plot
plt.savefig('Deletion_AlphaGenome_ChipHistone_H3K9ac.pdf')

In [ ]:
# Plotting histone chip seq - zoom in to CDKN2A/2B (H3K36me3)

transcripts = transcript_extractor.extract(interval)
mask5 = (ref_prediction.chip_histone.metadata.histone_mark == "H3K36me3").to_numpy()
mask6 = (alt_prediction.chip_histone.metadata.histone_mark == "H3K36me3").to_numpy()
# filter track
ref_h3k36me3 = ref_prediction.chip_histone.filter_tracks(mask5)
alt_h3k36me3 = alt_prediction.chip_histone.filter_tracks(mask6)

_ = plot_components.plot(
    [
        plot_components.TranscriptAnnotation(transcripts, fig_height=2),
        plot_components.Tracks(
            tdata=alt_h3k36me3 - ref_h3k36me3,
            ylabel_template="{biosample_name} ({strand})\n{name}",
            filled=True,
            shared_y_scale=True,
            track_colors="#8172B2",
        ),
    ],
    interval=genome.Interval(
        chromosome="chr9", start=21965000, end=22015000
    ),
    
    title="ChIP Histone: H3K36me3 alt - ref (predict_sequence)",
)
# Save the plot
plt.savefig('Deletion_AlphaGenome_ChipHistone_H3K36me3.pdf')

In [ ]:
# Plotting histone chip seq - zoom in to CDKN2A/2B (H3K27ac)

transcripts = transcript_extractor.extract(interval)
mask7 = (ref_prediction.chip_histone.metadata.histone_mark == "H3K27ac").to_numpy()
mask8 = (alt_prediction.chip_histone.metadata.histone_mark == "H3K27ac").to_numpy()
# filter track
ref_h3k27ac = ref_prediction.chip_histone.filter_tracks(mask7)
alt_h3k27ac = alt_prediction.chip_histone.filter_tracks(mask8)

_ = plot_components.plot(
    [
        plot_components.TranscriptAnnotation(transcripts, fig_height=2),
        plot_components.Tracks(
            tdata=alt_h3k27ac - ref_h3k27ac,
            ylabel_template="{biosample_name} ({strand})\n{name}",
            filled=True,
            shared_y_scale=True,
            track_colors="#55A868",
        ),
    ],
    interval=genome.Interval(
        chromosome="chr9", start=21965000, end=22015000
    ),
    title="ChIP Histone: H3K27ac alt - ref (predict_sequence)",
)
# Save the plot
plt.savefig('Deletion_AlphaGenome_ChipHistone_H3K27ac.pdf')

In [ ]:
# Save RNAseq output 
# Note: RNAseq prediction is 1bp resolution
rna_ref = ref_prediction.rna_seq  # from predict_sequence
rna_alt = alt_prediction.rna_seq

# 1. Extract values
ref_values = rna_ref.values  # shape: (positions, tracks)
alt_values = rna_alt.values

# 2. Track names (biosamples)
ref_names = list(rna_ref.metadata["name"])
alt_names = list(rna_alt.metadata["name"])

# 3. Build dataframe
df_ref = pd.DataFrame(ref_values, columns=ref_names)
df_alt = pd.DataFrame(alt_values, columns=alt_names)

# 4. Add genomic coordinates
start_ref = rna_ref.interval.start
chrom = rna_ref.interval.chromosome
start_alt = rna_alt.interval.start

df_ref.insert(0, "position", range(start_ref, start_ref + len(df_ref)))
df_ref.insert(0, "chr", chrom)

df_alt.insert(0, "position", range(start_alt, start_alt + len(df_alt)))
df_alt.insert(0, "chr", chrom)

# 5. Save
df_ref.to_csv("042826_alphagenome_rnaseq_Ref.csv", index=False)
df_alt.to_csv("042826_alphagenome_rnaseq_Alt.csv", index=False)
